# Feature extractors: basics

This tutorial introduces the `light_curve` feature extractor interface:
creating features, combining them with [`Extractor`](../api/meta/#light_curve.Extractor),
reading names and descriptions, and batch processing.

In [ ]:
# %pip install light-curve nested-pandas polars universal-pathlib

## Single feature

Each feature class is callable. It accepts `(t, m, sigma)` arrays and returns a NumPy array.
The `.names` attribute lists the output column names.

Here we use [`Amplitude`](../api/variability/#light_curve.Amplitude) — the half peak-to-peak amplitude:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(42)
t = np.sort(rng.uniform(0, 100, 200))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, 200)
err = np.full(200, 0.05)

amp = licu.Amplitude()
result = amp(t, m, err)
print(f'names:  {amp.names}')
print(f'result: {result}')

`.descriptions` gives a human-readable explanation of each output:

In [ ]:
import light_curve as licu

f = licu.EtaE()
print(f.descriptions)

## Combining features with `Extractor`

[`Extractor`](../api/meta/#light_curve.Extractor) combines multiple features into a single callable evaluated in one pass.
It is especially efficient for **cheap features** (statistical moments, variability
indices, etc.) because it avoids some computations and reduces Python–Rust
call overhead:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(42)
n = 200
t = np.sort(rng.uniform(0, 100, n))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.1, n)
err = np.full(n, 0.1)

ext = licu.Extractor(
    licu.InterPercentileRange(quantile=0.25),
    licu.BeyondNStd(nstd=1),
    licu.BeyondNStd(nstd=2),
    licu.StandardDeviation(),
    licu.WeightedMean(),
    licu.LinearFit(),
    licu.StetsonK(),
)
result = ext(t, m, err)
for name, value in zip(ext.names, result):
    print(f'  {name:35s} = {value:.5f}')

## Batch processing with `.many()`

`.many()` processes a list of `(t, m, sigma)` tuples and returns a 2-D NumPy array
(shape `(N, n_features)`). It supports **multi-threading** (enabled by default via the
`n_jobs` parameter) and is the preferred path for large datasets:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(0)
light_curves = [
    (np.sort(rng.random(50)), rng.random(50), rng.random(50) * 0.1)
    for _ in range(1000)
]

feature = licu.Extractor(licu.Skew(), licu.Kurtosis(), licu.ReducedChi2())
results = feature.many(light_curves)
print(f'Extracted from {len(light_curves)} light curves: shape = {results.shape}')
for name, col in zip(feature.names, results.T):
    print(f'  {name:20s} mean = {col.mean():.4f}')

## Batch processing with nested-pandas

[nested-pandas](https://nested-pandas.readthedocs.io) stores each light curve as a nested Arrow column,
letting `.many()` consume it with zero copies.
The `generate_data` helper creates a toy `NestedFrame` — its `nested` column holds
`t`, `flux`, and `band` fields:

In [ ]:
# %pip install light-curve nested-pandas

In [ ]:
import light_curve as licu
import pyarrow as pa
from nested_pandas.datasets import generate_data

ndf = generate_data(100, 50, seed=42)

feature = licu.Extractor(licu.ObservationCount(bands=["g", "r"]), licu.StandardDeviation(bands=["g", "r"]))
result = feature.many(pa.array(ndf["nested"]), arrow_fields={"t": "t", "m": "flux", "band": "band"})
ndf = ndf.assign(**dict(zip(feature.names, result.T)))
ndf[["a", "b", *feature.names]].head()

## Multiband light curves

Every feature accepts a `bands=` constructor argument to switch into **per-passband mode**.
When set, `__call__` expects a fourth `band` string array; outputs are named with a passband
suffix (e.g. `amplitude_g`, `amplitude_r`).

`Extractor` freely mixes single-band and multiband features — it filters the `band` array
automatically for each sub-feature:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(42)
t = np.sort(rng.uniform(0, 100, 200))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, 200)
err = np.full(200, 0.05)
band = np.tile(["g", "r"], 100)

sk = licu.StetsonK(bands=["g", "r"])
print("StetsonK per band:", dict(zip(sk.names, sk(t, m, err, band))))

ext = licu.Extractor(
    licu.EtaE(bands=["g", "r"]),
    licu.LinearFit(bands=["g", "r"]),
    licu.WeightedMean(),
)
result = ext(t, m, err, band)
for name, val in zip(ext.names, result):
    print(f"  {name:35s} = {val:.4f}")

## Next steps

- [Feature table](../) — all 40+ extractors grouped by category
- [API reference](../api/) — full signatures and equations
- [Periodogram tutorial](../periodogram/) — Lomb–Scargle and period search
- [Multiband tutorial](../multiband/) — per-band and cross-band features
- [Rainbow fit tutorial](../multiband/rainbow/) — blackbody temperature and radius evolution
- [Batch processing tutorial](batch_processing.ipynb) — nested-pandas with real survey data, PyArrow, Polars